In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from model.autoencoder import Model
from train import Trainer
from data import get_cross_data, get_rec_data, load_data

In [ ]:
batch_size = 32
N = batch_size  # Batch size
T = 64          # Number of frames (64)
M = 1           # Number of persons
V = 25          # Number of joints
C_in = 3        # Number of input channels
setting = 'cs'  # 'cs' or 'cv'
dataset = 'ntu120'

In [ ]:
X = load_data(dataset)
paired_train, paired_test = get_cross_data(X, dataset, setting, batch_size, T, return_loader=True, train_samples=64, test_samples=32)#train_samples=50016, test_samples=5024)
train, test = get_rec_data(X, dataset, setting, T, batch_size,)

In [ ]:
# Initialize the model
model = Model(num_class=120, num_point=25, num_person=1, graph='graph.ntu_rgb_d.Graph',
              graph_args={'labeling_mode': 'spatial'}, debug=False)
model = model.cuda()

# Define optimizer and loss criterion
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.MSELoss()

# Number of epochs for each stage
num_epochs_stage1 = 1
num_epochs_stage2 = 1

# Create Trainer instance
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    train_loader=train,
    val_loader=test,
    train_paired_loader=paired_train,
    val_paired_loader=paired_test,
    num_epochs_stage1=num_epochs_stage1,
    num_epochs_stage2=num_epochs_stage2,
    device='cuda'  # or 'cpu' if not using GPU
)

# Train Stage 1
trainer.train_stage1()

# Train Stage 2
trainer.train_stage2()